<a href="https://colab.research.google.com/github/Jesus-0655/Satelites_Actividad_IA_LITIID74/blob/main/AstroShield_LITIID74_Equipo_Dinamita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# AstroShield — Motor Lógico y Optimización de la Constelación
Aqui esta toda la lógica matemática del modelo planetario, el sistema de coordenadas,
el cálculo de cobertura por muestreo, el mecanismo de reparación de restricciones y el algoritmo evolutivo completo.
```

In [ ]:
import numpy as np
import math
import time

# --- CONSTANTES GLOBALES Y MODELO ---
R_EARTH = 6371.0 # Radio de la Tierra en km
LAMBDA_DEG = 13.4 # Radio de cobertura del satélite en grados (~1490 km)
LIMIT_SEPARATION_DEG = 5.0 # Separación mínima entre satélites

# --- DATOS DE REGIONES DE INTERÉS ---
REGIONS = [
    {"name": "Península de Yucatán, México", "lat": 20.5, "lon": -89.0, "priority_weight": 3},
    {"name": "Amazonía brasileña", "lat": -3.5, "lon": -62.0, "priority_weight": 3},
    {"name": "California, Estados Unidos", "lat": 36.8, "lon": -119.4, "priority_weight": 2},
    {"name": "Región del Sahel, África", "lat": 15.0, "lon": 0.0, "priority_weight": 2},
    {"name": "Japón", "lat": 36.0, "lon": 138.0, "priority_weight": 2},
    {"name": "Interior de Australia", "lat": -25.0, "lon": 133.0, "priority_weight": 1}
]
TOTAL_PRIORITY_WEIGHT = sum(r["priority_weight"] for r in REGIONS) # W = 13

# --- RESTRICCIONES ESPACIALES ---
FORBIDDEN_ZONES = [
    # Zonas de desechos espaciales
    {"name": "Atlántico Norte", "lat": 45.0, "lon": -35.0, "radius": 12.0},
    {"name": "Pacífico Sur", "lat": -30.0, "lon": -120.0, "radius": 10.0},
    {"name": "Región polar norte", "lat": 78.0, "lon": 30.0, "radius": 8.0},
    # Posiciones ocupadas
    {"name": "Objeto A", "lat": 10.0, "lon": 45.0, "radius": 5.0},
    {"name": "Objeto B", "lat": -15.0, "lon": 110.0, "radius": 5.0}
]

# --- FUNCIONES GEOMÉTRICAS ---
def haversine_distance_deg(lat1, lon1, lat2, lon2):
    """Calcula la distancia angular en grados entre dos coordenadas esféricas."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(delta_lambda/2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return math.degrees(c)

def wrap_longitude(lon):
    """Ajusta la longitud al rango [-180, 180]."""
    return (lon + 180.0) % 360.0 - 180.0

def sample_sphere_point_uniform():
    """Genera un punto uniforme en la esfera usando arcsin para evitar acumulación en polos."""
    lon = np.random.uniform(-180.0, 180.0)
    lat = math.degrees(math.asin(np.random.uniform(-1.0, 1.0)))
    return lat, lon

def generate_region_samples(center_lat, center_lon, radius_km=300.0):
    """Genera 9 puntos de muestreo por región (centro + 8 perimetrales a ~2.70°)."""
    points = [(center_lat, center_lon)]
    angular_radius_deg = math.degrees(radius_km / R_EARTH) # ~2.697 grados
    for i in range(8):
        angle = i * (2.0 * math.pi / 8.0)
        d_lat = angular_radius_deg * math.cos(angle)
        # Ajuste de longitud por la deformación del paralelo
        cos_lat = math.cos(math.radians(center_lat))
        d_lon = angular_radius_deg * math.sin(angle) / cos_lat if cos_lat > 1e-6 else 0.0
        points.append((np.clip(center_lat + d_lat, -90.0, 90.0), wrap_longitude(center_lon + d_lon)))
    return points

# Precomputamos las muestras de cada región para acelerar el fitness
for r in REGIONS:
    r["samples"] = generate_region_samples(r["lat"], r["lon"])